# 01: Translation run

The **only** notebook that calls the LLM. Each matrix cell (target x model) has its own code cell below, so one (target, model) pair can be run or re-run in isolation and its per-query results inspected immediately. Records go to `records_<dataset>_<target>_<model>.json` in `eval/outputs/records/`; downstream notebooks glob those files.

Configuration lives in `harness.config` (temperature 0.0, max 3 iterations, offline syntax validation). Token counts come straight from `result.token_usage` -- no log scraping. Resume is automatic: query ids already on disk for a cell are skipped; delete a cell's records file to force a full re-run. Run the cells serially, top to bottom; never run model cells in parallel.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "eval"))

from dataclasses import replace

import pandas as pd

from harness import RECORDS_DIR, RUN_MATRIX, billed_input_tokens, load_records, run_translation

print(f'{len(RUN_MATRIX)} matrix cell(s):')
for rc in RUN_MATRIX:
    print(f'  - {rc.dataset} x {rc.target} x {rc.model} ({rc.provider})')

12 matrix cell(s):
  - ldbc x cypher x llama3.2:latest (ollama)
  - ldbc x cypher x qwen3-coder:30b (ollama)
  - ldbc x cypher x gemma4:26b (ollama)
  - ldbc x cypher x claude-opus-4-8 (anthropic)
  - ldbc x aql x llama3.2:latest (ollama)
  - ldbc x aql x qwen3-coder:30b (ollama)
  - ldbc x aql x gemma4:26b (ollama)
  - ldbc x aql x claude-opus-4-8 (anthropic)
  - ldbc x gremlin x llama3.2:latest (ollama)
  - ldbc x gremlin x qwen3-coder:30b (ollama)
  - ldbc x gremlin x gemma4:26b (ollama)
  - ldbc x gremlin x claude-opus-4-8 (anthropic)


## Helpers and smoke subset

`run_cell(target, model)` runs one matrix cell and returns its per-query results table; `summarize_target(target)` aggregates one target's records on disk by model. Set `SUBSET` to a tuple of query ids (e.g. `('ldbc_q01',)`) to restrict every cell to a smoke subset before paying for a full run; `None` runs the whole gold set.

In [2]:
SUBSET: tuple[str, ...] | None = None  # e.g. ('ldbc_q01',) for a smoke run

RESULT_COLS = ['query_id', 'difficulty', 'validation_passed', 'iterations_used', 'status',
               'billed_input_tokens', 'output_tokens', 'duration_seconds', 'error']

def find_cell(target, model):
    """The unique RUN_MATRIX cell for (target, model)."""
    matches = [rc for rc in RUN_MATRIX if rc.target == target and rc.model == model]
    assert len(matches) == 1, f'{len(matches)} matrix cell(s) for {target}/{model}, expected 1'
    return matches[0]

def run_cell(target, model):
    """Run one matrix cell (serially); returns its per-query results table."""
    rc = find_cell(target, model)
    if SUBSET is not None:
        rc = replace(rc, subset=SUBSET)
    df = pd.DataFrame(run_translation(rc))
    if not len(df):
        print('No records.')
        return df
    df['billed_input_tokens'] = billed_input_tokens(df['input_tokens'], df['cache_read_tokens'], df['cache_creation_tokens'])
    return df[RESULT_COLS]

def summarize_target(target):
    """Aggregate this target's records on disk, by model."""
    df = pd.DataFrame(load_records(RECORDS_DIR, target=target))
    if not len(df):
        print(f'No records for {target} yet.')
        return None
    df['billed_input_tokens'] = billed_input_tokens(df['input_tokens'], df['cache_read_tokens'], df['cache_creation_tokens'])
    return df.groupby('model').agg(
        n=('query_id', 'count'), passed=('validation_passed', 'sum'),
        mean_iterations=('iterations_used', 'mean'),
        in_tok=('billed_input_tokens', 'sum'), out_tok=('output_tokens', 'sum'),
        mean_duration_s=('duration_seconds', 'mean'))

print(f'SUBSET = {SUBSET}')

SUBSET = None


## SQL → Cypher

### llama3.2:latest

In [3]:
run_cell('cypher', 'llama3.2:latest')

Resume: 15 record(s) on disk for records_ldbc_cypher_llama3.2_latest.json; 15 query id(s) done.
ldbc/cypher/llama3.2:latest: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_cypher_llama3.2_latest.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2404,35,6.945831,None
1,ldbc_q02,easy,True,3,success,8910,162,13.532050,None
2,ldbc_q03,easy,True,1,success,2401,185,8.171838,None
3,ldbc_q04,hard,True,1,success,3172,193,9.199575,None
4,ldbc_q05,hard,True,1,success,3461,81,4.061995,None
5,ldbc_q06,medium,True,1,success,2732,69,3.304163,None
6,ldbc_q07,medium,True,1,success,3199,104,5.110439,None
7,ldbc_q08,hard,True,3,success,9238,421,20.374309,None
8,ldbc_q09,medium,True,1,success,2821,85,3.901266,None
9,ldbc_q10,hard,True,1,success,3173,181,8.464310,None


### qwen3-coder:30b

In [4]:
run_cell('cypher', 'qwen3-coder:30b')

Resume: 15 record(s) on disk for records_ldbc_cypher_qwen3-coder_30b.json; 15 query id(s) done.
ldbc/cypher/qwen3-coder:30b: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_cypher_qwen3-coder_30b.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2406,31,17.392788,None
1,ldbc_q02,easy,True,1,success,2890,71,19.496970,None
2,ldbc_q03,easy,True,1,success,2401,22,1.185505,None
3,ldbc_q04,hard,True,1,success,3175,49,3.422444,None
4,ldbc_q05,hard,True,1,success,3464,83,4.775432,None
5,ldbc_q06,medium,True,1,success,2734,47,3.332481,None
6,ldbc_q07,medium,True,1,success,3232,98,5.407516,None
7,ldbc_q08,hard,True,1,success,2841,93,5.113587,None
8,ldbc_q09,medium,True,1,success,2831,65,3.474695,None
9,ldbc_q10,hard,True,1,success,3175,42,4.065331,None


### gemma4:26b

In [5]:
run_cell('cypher', 'gemma4:26b')

Resume: 15 record(s) on disk for records_ldbc_cypher_gemma4_26b.json; 15 query id(s) done.
ldbc/cypher/gemma4:26b: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_cypher_gemma4_26b.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2735,422,37.736048,None
1,ldbc_q02,easy,True,1,success,3262,1215,56.467919,None
2,ldbc_q03,easy,True,1,success,2730,415,5.185734,None
3,ldbc_q04,hard,True,1,success,3563,1024,12.430539,None
4,ldbc_q05,hard,True,1,success,3864,2099,24.740709,None
5,ldbc_q06,medium,True,1,success,3094,1236,15.012691,None
6,ldbc_q07,medium,True,1,success,3642,1904,22.617156,None
7,ldbc_q08,hard,True,1,success,3222,2134,24.555018,None
8,ldbc_q09,medium,True,1,success,3203,2013,23.373235,None
9,ldbc_q10,hard,True,1,success,3566,828,10.780821,None


### claude-opus-4-8

In [6]:
run_cell('cypher', 'claude-opus-4-8')

Resume: 15 record(s) on disk for records_ldbc_cypher_claude-opus-4-8.json; 15 query id(s) done.
ldbc/cypher/claude-opus-4-8: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_cypher_claude-opus-4-8.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3825,50,2.084392,None
1,ldbc_q02,easy,True,1,success,4566,87,7.168288,None
2,ldbc_q03,easy,True,1,success,3823,45,1.633420,None
3,ldbc_q04,hard,True,1,success,5186,76,2.711763,None
4,ldbc_q05,hard,True,1,success,5709,96,2.775870,None
5,ldbc_q06,medium,True,1,success,4385,73,2.871345,None
6,ldbc_q07,medium,True,1,success,5152,140,2.847695,None
7,ldbc_q08,hard,True,1,success,4592,151,2.855862,None
8,ldbc_q09,medium,True,1,success,4572,118,3.690251,None
9,ldbc_q10,hard,True,1,success,5188,89,2.324540,None


### Aggregation by model

In [7]:
summarize_target('cypher')

,n,passed,mean_iterations,in_tok,out_tok,mean_duration_s
model,,,,,,
claude-opus-4-8,15,15,1.000000,70926,1413,2.960200
gemma4:26b,15,15,1.000000,49506,25596,32.368910
llama3.2:latest,15,15,1.266667,56264,2148,8.117859
qwen3-coder:30b,15,15,1.000000,43926,968,8.400243


## SQL → AQL

### llama3.2:latest

In [8]:
run_cell('aql', 'llama3.2:latest')

Resume: 15 record(s) on disk for records_ldbc_aql_llama3.2_latest.json; 15 query id(s) done.
ldbc/aql/llama3.2:latest: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_aql_llama3.2_latest.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3084,110,10.834274,None
1,ldbc_q02,easy,True,1,success,3412,163,8.007973,None
2,ldbc_q03,easy,True,1,success,3081,124,5.935551,None
3,ldbc_q04,hard,True,2,success,8603,329,16.883880,None
4,ldbc_q05,hard,True,2,success,9219,333,17.596494,None
5,ldbc_q06,medium,True,1,success,3462,198,9.611270,None
6,ldbc_q07,medium,True,1,success,3806,213,10.314666,None
7,ldbc_q08,hard,True,3,success,12952,746,37.362195,None
8,ldbc_q09,medium,False,3,max_iterations_reached,11500,465,23.329497,None
9,ldbc_q10,hard,True,2,success,8779,555,28.752213,None


### qwen3-coder:30b

In [9]:
run_cell('aql', 'qwen3-coder:30b')

Resume: 15 record(s) on disk for records_ldbc_aql_qwen3-coder_30b.json; 15 query id(s) done.
ldbc/aql/qwen3-coder:30b: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_aql_qwen3-coder_30b.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3088,49,18.903891,None
1,ldbc_q02,easy,True,1,success,3441,61,3.055459,None
2,ldbc_q03,easy,True,1,success,3083,35,1.827960,None
3,ldbc_q04,hard,True,1,success,4018,88,6.200545,None
4,ldbc_q05,hard,True,1,success,4334,93,5.246304,None
5,ldbc_q06,medium,True,1,success,3466,55,4.275656,None
6,ldbc_q07,medium,True,1,success,3835,104,6.019845,None
7,ldbc_q08,hard,True,1,success,3601,113,6.641182,None
8,ldbc_q09,medium,True,1,success,3623,79,4.078746,None
9,ldbc_q10,hard,True,1,success,4018,51,4.485285,None


### gemma4:26b

In [10]:
run_cell('aql', 'gemma4:26b')

Resume: 15 record(s) on disk for records_ldbc_aql_gemma4_26b.json; 15 query id(s) done.
ldbc/aql/gemma4:26b: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_aql_gemma4_26b.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,3441,453,21.159219,None
1,ldbc_q02,easy,True,1,success,3818,1225,14.435183,None
2,ldbc_q03,easy,True,1,success,3436,429,5.401434,None
3,ldbc_q04,hard,True,1,success,4432,1981,24.457997,None
4,ldbc_q05,hard,True,1,success,4764,6385,72.695676,None
5,ldbc_q06,medium,True,1,success,3856,1801,22.087202,None
6,ldbc_q07,medium,True,1,success,4258,2302,26.812581,None
7,ldbc_q08,hard,True,2,success,8617,11571,133.419983,None
8,ldbc_q09,medium,True,1,success,4026,5565,64.891218,None
9,ldbc_q10,hard,True,1,success,4435,1447,18.305811,None


### claude-opus-4-8

In [11]:
run_cell('aql', 'claude-opus-4-8')

Resume: 15 record(s) on disk for records_ldbc_aql_claude-opus-4-8.json; 15 query id(s) done.
ldbc/aql/claude-opus-4-8: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_aql_claude-opus-4-8.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,5049,77,3.278285,None
1,ldbc_q02,easy,True,1,success,5619,80,2.613209,None
2,ldbc_q03,easy,True,1,success,5047,62,4.596436,None
3,ldbc_q04,hard,True,1,success,6693,104,2.334190,None
4,ldbc_q05,hard,True,1,success,7275,212,3.771293,None
5,ldbc_q06,medium,True,1,success,5682,100,2.337970,None
6,ldbc_q07,medium,True,1,success,6278,142,2.853055,None
7,ldbc_q08,hard,True,1,success,5944,216,2.959778,None
8,ldbc_q09,medium,True,1,success,5971,142,2.539799,None
9,ldbc_q10,hard,True,1,success,6695,95,2.238497,None


### Aggregation by model

In [12]:
summarize_target('aql')

,n,passed,mean_iterations,in_tok,out_tok,mean_duration_s
model,,,,,,
claude-opus-4-8,15,15,1.000000,91382,1913,3.060915
gemma4:26b,15,15,1.066667,65868,46451,38.112693
llama3.2:latest,15,12,1.800000,109403,5193,21.446847
qwen3-coder:30b,15,15,1.000000,55295,1152,7.132720


## SQL → Gremlin

### llama3.2:latest

In [13]:
run_cell('gremlin', 'llama3.2:latest')

Resume: 15 record(s) on disk for records_ldbc_gremlin_llama3.2_latest.json; 15 query id(s) done.
ldbc/gremlin/llama3.2:latest: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_gremlin_llama3.2_latest.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2671,184,13.159977,None
1,ldbc_q02,easy,False,3,max_iterations_reached,9857,662,30.228762,None
2,ldbc_q03,easy,True,1,success,2668,164,7.280943,None
3,ldbc_q04,hard,False,3,max_iterations_reached,11898,725,36.715658,None
4,ldbc_q05,hard,False,3,max_iterations_reached,12070,820,41.049272,None
5,ldbc_q06,medium,False,3,max_iterations_reached,9717,369,17.293283,None
6,ldbc_q07,medium,False,3,max_iterations_reached,11101,758,36.326841,None
7,ldbc_q08,hard,True,1,success,3183,698,32.553665,None
8,ldbc_q09,medium,False,3,max_iterations_reached,11551,792,37.760254,None
9,ldbc_q10,hard,False,3,max_iterations_reached,11047,613,29.893580,None


### qwen3-coder:30b

In [14]:
run_cell('gremlin', 'qwen3-coder:30b')

Resume: 15 record(s) on disk for records_ldbc_gremlin_qwen3-coder_30b.json; 15 query id(s) done.
ldbc/gremlin/qwen3-coder:30b: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_gremlin_qwen3-coder_30b.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2675,56,11.609653,NaN
1,ldbc_q02,easy,True,1,success,2992,72,2.872458,NaN
2,ldbc_q03,easy,True,1,success,2670,41,1.913921,NaN
3,ldbc_q04,hard,True,3,success,10842,340,18.067965,NaN
4,ldbc_q05,hard,True,1,success,3670,109,5.568143,NaN
5,ldbc_q06,medium,True,1,success,3045,58,3.427081,NaN
6,ldbc_q07,medium,True,1,success,3377,93,4.488420,NaN
7,ldbc_q08,hard,False,1,generation_hang,0,0,NaN,ollama generation hung on this prompt in three...
8,ldbc_q09,medium,True,1,success,3203,108,5.338304,NaN
9,ldbc_q10,hard,True,1,success,3387,94,5.841224,NaN


### gemma4:26b

In [15]:
run_cell('gremlin', 'gemma4:26b')

Resume: 15 record(s) on disk for records_ldbc_gremlin_gemma4_26b.json; 15 query id(s) done.
ldbc/gremlin/gemma4:26b: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_gremlin_gemma4_26b.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,2998,1239,24.769073,None
1,ldbc_q02,easy,True,1,success,3322,829,10.268602,None
2,ldbc_q03,easy,True,1,success,2993,697,8.568272,None
3,ldbc_q04,hard,True,1,success,3758,1962,23.684939,None
4,ldbc_q05,hard,True,1,success,4055,5006,58.420987,None
5,ldbc_q06,medium,True,1,success,3393,2372,28.340513,None
6,ldbc_q07,medium,True,1,success,3738,2995,34.905015,None
7,ldbc_q08,hard,True,1,success,3556,3696,50.942750,None
8,ldbc_q09,medium,True,1,success,3570,4642,63.552989,None
9,ldbc_q10,hard,True,1,success,3761,2985,35.526123,None


### claude-opus-4-8

In [16]:
run_cell('gremlin', 'claude-opus-4-8')

Resume: 15 record(s) on disk for records_ldbc_gremlin_claude-opus-4-8.json; 15 query id(s) done.
ldbc/gremlin/claude-opus-4-8: 0 item(s) to translate.
Done: 15 record(s) in /Users/ivona.obonova/school/sql2graph/sql2graph/eval/outputs/records/records_ldbc_gremlin_claude-opus-4-8.json


,query_id,difficulty,validation_passed,iterations_used,status,billed_input_tokens,output_tokens,duration_seconds,error
0,ldbc_q01,easy,True,1,success,4127,76,2.570988,None
1,ldbc_q02,easy,True,1,success,4571,85,1.942554,None
2,ldbc_q03,easy,True,1,success,4125,61,2.109440,None
3,ldbc_q04,hard,True,1,success,5261,78,2.800566,None
4,ldbc_q05,hard,True,1,success,5724,78,4.037739,None
5,ldbc_q06,medium,True,1,success,4683,81,2.037096,None
6,ldbc_q07,medium,True,1,success,5153,186,4.283523,None
7,ldbc_q08,hard,True,1,success,4926,141,4.017482,None
8,ldbc_q09,medium,True,1,success,4933,157,3.045629,None
9,ldbc_q10,hard,True,1,success,5263,73,2.142205,None


### Aggregation by model

In [17]:
summarize_target('gremlin')

,n,passed,mean_iterations,in_tok,out_tok,mean_duration_s
model,,,,,,
claude-opus-4-8,15,15,1.000000,73826,1383,2.732615
gemma4:26b,15,15,1.066667,57192,46426,38.251002
llama3.2:latest,15,3,2.600000,140396,9424,30.926102
qwen3-coder:30b,15,11,1.400000,64911,2006,8.829270


## Run-level summary

In [18]:

df = pd.DataFrame(load_records(RECORDS_DIR))
if len(df):
    df['billed_input_tokens'] = billed_input_tokens(df['input_tokens'], df['cache_read_tokens'], df['cache_creation_tokens'])
    print(f'Total records: {len(df)}')
    print(f"Validation passed: {int(df['validation_passed'].sum())} / {len(df)}")
    print(f"Total tokens: {int(df['billed_input_tokens'].sum()):,} in / {int(df['output_tokens'].sum()):,} out")
    print(f"Mean duration: {df['duration_seconds'].mean():.2f}s")
    display(df.groupby(['dataset','target','model']).agg(
        n=('query_id','count'), passed=('validation_passed','sum'),
        mean_iter=('iterations_used','mean'),
        in_tok=('billed_input_tokens','sum'), out_tok=('output_tokens','sum')))
else:
    print('No records yet.')

Total records: 180
Validation passed: 161 / 180
Total tokens: 878,895 in / 144,073 out
Mean duration: 16.95s


n  passed  mean_iter  in_tok  out_tok
dataset target  model                                                  
ldbc    aql     claude-opus-4-8  15      15   1.000000   91382     1913
                gemma4:26b       15      15   1.066667   65868    46451
                llama3.2:latest  15      12   1.800000  109403     5193
                qwen3-coder:30b  15      15   1.000000   55295     1152
        cypher  claude-opus-4-8  15      15   1.000000   70926     1413
                gemma4:26b       15      15   1.000000   49506    25596
                llama3.2:latest  15      15   1.266667   56264     2148
                qwen3-coder:30b  15      15   1.000000   43926      968
        gremlin claude-opus-4-8  15      15   1.000000   73826     1383
                gemma4:26b       15      15   1.066667   57192    46426
                llama3.2:latest  15       3   2.600000  140396     9424
                qwen3-coder:30b  15      11   1.400000   64911     2006

Records written. Proceed to `02_behavioural_metrics.ipynb`.